# Student Exam Performance Analysis & Machine Learning Exploration

## 1. Project Background & Business Context
Educational institutions frequently seek actionable indicators to identify students who may require additional academic support before final examinations occur. By understanding how factors such as study time, attendance, sleep habits, and tutoring impact academic outcomes, schools can deploy targeted interventions early.

This notebook tells the story of our student dataset: investigating distributions, checking data hygiene, identifying correlations, and formulating a hypothesis-driven machine learning strategy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load the dataset
data_path = Path('../data/student_data.csv')
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Data Overview & Initial Observations
The dataset contains 1,000 student records with 8 predictive features:
- **Numerical features:** `hours_studied`, `attendance_percent`, `previous_score`, `sleep_hours`, `tutoring_sessions`
- **Categorical features:** `parental_involvement` (ordinal: Low/Medium/High), `extracurricular` (binary: Yes/No), `internet_access` (binary: Yes/No)
- **Target variable:** `final_score` (continuous exam score bounded from 0 to 100)

Next, we conduct data sanity checks to examine missing values and statistical properties.

In [ ]:
# Summary statistics and missing value check
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
data_quality_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Percent': missing_pct})
print("=== Missing Value Analysis ===")
print(data_quality_df[data_quality_df['Missing_Count'] > 0])

print("\n=== Descriptive Statistics (Numerical Features) ===")
df.describe().round(2)

### Data Hygiene & Imputation Strategy
- **Missing Value Findings:** As seen above, missingness is below 2% and isolated to `sleep_hours`, `tutoring_sessions`, and `parental_involvement`. The target `final_score` has 0 missing values.
- **Handling Strategy:** Dropping rows would needlessly discard statistical power. Following ML best practices, we will use median imputation for numerical features (robust against skew) and most frequent value for categorical features inside an isolated preprocessing pipeline.

In [ ]:
# Visualizing the Target Variable Distribution
plt.figure(figsize=(8, 4.5))
plt.hist(df['final_score'], bins=25, color='#3b82f6', edgecolor='black', alpha=0.7)
plt.axvline(df['final_score'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['final_score'].mean():.1f}")
plt.axvline(df['final_score'].median(), color='green', linestyle=':', linewidth=2, label=f"Median: {df['final_score'].median():.1f}")
plt.title('Distribution of Student Final Exam Scores', fontsize=12, fontweight='bold')
plt.xlabel('Final Score (0-100)')
plt.ylabel('Student Count')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

### Target Distribution Insights
- The exam score distribution approximates a bell-curve (normal distribution) centered around ~75 points.
- Because the target is smoothly distributed without extreme bi-modality, standard regression metrics like **Root Mean Squared Error (RMSE)** and **Mean Absolute Error (MAE)** will provide reliable feedback during model optimization.

In [ ]:
# Feature Correlation with Final Score
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

print("Correlations with Final Score:")
print(corr_matrix['final_score'].sort_values(ascending=False).round(3))

fig, ax = plt.subplots(figsize=(7, 5.5))
cax = ax.matshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax)
ticks = np.arange(len(numeric_cols))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(numeric_cols, rotation=45, ha='left')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha='center', va='center', color='black')
plt.title('Feature Correlation Matrix', pad=40, fontweight='bold')
plt.show()

### Correlation Analysis & Hypothesis
- `previous_score`, `hours_studied`, and `attendance_percent` exhibit the strongest positive correlations with `final_score`.
- Moderate positive associations exist with `tutoring_sessions` and `sleep_hours`.
- Multicollinearity among independent features is low, suggesting that both linear estimators (Linear & Ridge Regression) and non-linear tree-based ensembles (Random Forest) should perform strongly.

## 2. Modeling Strategy & Conclusion
Based on our exploratory findings, we establish the following machine learning methodology:
1. **Strict Train/Val/Test Partitioning**: 70% training, 15% validation, 15% test. Preprocessing transformers will strictly be fit on training data alone to eliminate data leakage.
2. **ColumnTransformer Pipeline**: Standardize numerical features, apply Ordinal Encoding to parental involvement, and One-Hot encode binary flags.
3. **Model Benchmark**: Compare an interpretable baseline (`LinearRegression`), an L2-regularized model (`Ridge`), and an ensemble model (`RandomForestRegressor`).
4. **Production Deployment**: Save the champion pipeline as a standalone artifact for instant single-sample or batch inference.